# Parallel Execution with ToolPool

When you have a batch of inputs and multiple GPUs sitting idle, the obvious thing to do is to split the work across them. `ToolPool` does that automatically. You wrap your call in a `with ToolPool():` context, and it partitions the input across every available device, runs the pieces in parallel worker processes, and stitches the results back together in the original order.

`ToolPool` is the multi-GPU counterpart to `ToolInstance.persist()`. Reach for `persist()` when one worker on one GPU is enough to amortize the load cost across a batch. Reach for `ToolPool()` when you want to spread that same batch across several GPUs to cut wall-clock time. Both mechanisms cooperate; inside a `ToolPool` block, every worker stays warm for later calls in the same block, so you get the benefits of persistence automatically.

| Feature | What it does |
|---|---|
| **Transparent interception** | Any tool with an `iterable_input_field` is auto-partitioned. |
| **Cost-aware scheduling** | Items are distributed via LPT bin-packing using each tool's `item_cost()` estimate. |
| **Built-in persistence** | Workers stay alive across calls within the pool, so reloading happens at most once per GPU. |
| **Device auto-detection** | Discovers all visible GPUs, or accepts an explicit device list. |
| **Automatic dedup** | Duplicate items are computed once and expanded back to their original positions. |

<Note>
`ToolPool` is for **local inference** on hardware you control. If you are running via cloud inference, partitioning and parallelism are handled remotely, so `ToolPool` is not needed.
</Note>

---

## 1. Basic Usage (Auto-Detect GPUs)

The simplest form of `ToolPool` takes no arguments. Every GPU the process can see joins the pool, and any `run_*` call inside the block gets partitioned across all of them. Results come back in input order, so the multi-GPU execution is invisible to the caller.

In [ ]:
from proto_tools.tools.structure_prediction.esmfold import (
    run_esmfold, ESMFoldInput,
)
from proto_tools.utils.tool_pool import ToolPool

complexes = [[seq] for seq in sequences]  # 48 sequences

with ToolPool():
    output = run_esmfold(ESMFoldInput(complexes=complexes))

# Structures come back in the same order as the input
assert len(output.structures) == len(complexes)

<img src="assets/parallel-execution/auto.svg" alt="ToolPool fans a 48-sequence batch across 4 GPUs, 12 sequences per GPU" width="560">

On a box with four GPUs, the 48-sequence batch above lands as 12 sequences per GPU. Each worker loads the model once in parallel, runs its slice, and the pool assembles the structures back into the original order before returning.

---

## 2. Restricting or Choosing Devices

Sometimes you do not want to use every visible GPU. Another job might be running on a couple of the cards, or you might want to benchmark a fixed number of devices. Pass `gpus=[...]` to restrict the pool explicitly.

In [ ]:
with ToolPool(gpus=["cuda:0", "cuda:1"]):
    output = run_esmfold(ESMFoldInput(complexes=complexes))

<img src="assets/parallel-execution/devices.svg" alt="Passing gpus=[...] restricts the ToolPool; unlisted GPUs stay free for other workloads" width="560">

Only the listed GPUs join the pool, so the same batch now partitions across two workers instead of four. The unlisted GPUs stay free for other work on the same machine.

---

## 3. Persistence Within a Pool

Once a worker loads inside a `ToolPool` block, it stays resident for every later call in that same block. The first call pays model-loading time on every GPU in the pool; every later call skips the load and runs against workers that are already warm.

In [ ]:
with ToolPool(gpus=["cuda:0", "cuda:1"]):
    # Cold call: pays model loading on both GPUs
    result1 = run_esmfold(ESMFoldInput(complexes=complexes))

    # other tasks here ...

    # Warm call: workers already loaded
    result2 = run_esmfold(ESMFoldInput(complexes=complexes))

    # other tasks here ...

    result3 = run_esmfold(ESMFoldInput(complexes=complexes))

<img src="assets/parallel-execution/persist.svg" alt="Workers stay resident across calls within the same ToolPool block; only the first call pays the load cost" width="560">

On exit, all workers are shut down and GPU memory is released. The pool takes care of the lifecycle for you.

---

## 4. Cost-Aware Scheduling

Real batches rarely contain items of uniform cost. One protein sequence might be 40 residues, another 800; a longer sequence takes proportionally more compute, so the long one dominates its partition. If you hand items out to GPUs round-robin, whichever worker happened to get the long items becomes the bottleneck while the others sit idle.

`ToolPool` avoids this with **Longest-Processing-Time-first (LPT) bin-packing**. Each tool reports a per-item cost estimate (for example, total residue count for structure prediction). `ToolPool` sorts the batch by descending cost and assigns each item to whichever worker currently has the least total work. The result is that every GPU finishes at roughly the same wall-clock time, regardless of how the input sizes are distributed.

<img src="assets/parallel-execution/lpt.svg" alt="Naive round-robin leaves GPUs idle while one bottlenecks; LPT balances finish times across all GPUs" width="560">

You do not need to do anything special to benefit from this. As long as the tool's `item_cost()` is reasonable, a mixed batch of short and long inputs will be balanced automatically. For a batch of all-equal-sized inputs, scheduling reduces to simple round-robin.

---

## 5. Automatic Deduplication

If the same input shows up multiple times in a batch, `ToolPool` only computes it once and expands the result back to every position where it appeared. This happens transparently, so the returned list always has the same length and order as the input.

In [ ]:
# 1000 calls, but only 3 unique sequences
sequences = (["MKTLLILAVVAAALA"] * 400
             + ["GAVLTVLLGGLLLA"] * 300
             + ["MGQQPGKVLGDQRR"] * 300)

with ToolPool():
    output = run_esmfold(ESMFoldInput(complexes=[[s] for s in sequences]))

# Only 3 folds were actually run; output.structures has 1000 entries

This matters most for sweeps and sampling schemes that produce many duplicates of a small set of distinct inputs. Without dedup, you would pay to fold the same structure hundreds of times for no reason.

---

## 6. When to Use ToolPool vs. ToolInstance.persist()

`ToolPool` and `ToolInstance.persist()` solve related but different problems. A quick decision table:

| Situation | Use |
|---|---|
| One GPU, batch of calls | [`ToolInstance.persist()`](/tools/guides/tool-persistence) |
| Multiple GPUs, single large batch | `ToolPool()` |
| Multiple GPUs, you want manual control over which tool runs where | [`ToolInstance.persist_tool(instance_name=...)`](/tools/guides/tool-persistence) |
| Cloud inference | Neither; the provider handles it |

One caveat: `ToolPool` only accelerates tools that declare an `iterable_input_field` (for example, `complexes` for ESMFold, `sequences` for ESM2). A tool that takes a single indivisible input will run on one GPU regardless of the pool size, because there is nothing to partition.

---

## Configuration Reference

In [ ]:
from proto_tools.utils.tool_pool import ToolPool

# Auto-detect every visible GPU
with ToolPool():
    ...

# Explicit device list
with ToolPool(gpus=["cuda:0", "cuda:2"]):
    ...

For implementation details (LPT scheduling, cost estimation, worker lifecycle), see `proto_tools/utils/tool_pool.py`.

## Next Steps

<CardGroup cols={2}>
  <Card title="Tool Persistence" icon="floppy-disk" href="/tools/guides/tool-persistence">
    Keep a single tool warm across many calls on one GPU.
  </Card>
  <Card title="Device Management" icon="microchip" href="/tools/guides/device-management">
    Pin specific tools to specific GPUs.
  </Card>
  <Card title="Cloud Inference" icon="cloud" href="/tools/guides/cloud-inference">
    Offload execution to managed remote compute.
  </Card>
  <Card title="Tool Environments" icon="box" href="/tools/guides/tool-environments">
    How isolated per-tool environments are built and cached.
  </Card>
</CardGroup>